# Lecture 01: Computational Research Workflows

**PHYS690: Computational Methods for Physics Research**  
**Thursday, August 27, 2026**

Today is a tour of the kind of work we will do this semester: turning a physics question into code, data, figures, checks, and a result that someone else can reproduce.

## How to use this workbook

This notebook is designed to run in [Google Colaboratory](https://colab.research.google.com/). It does not need any external files. All data are generated inside the notebook.

To launch the workbook on Google Colaboratory, click this [link](https://drive.google.com/file/d/1bcdrrXhBndr6--Vg6vlxQqmHYv-vZ8pX/view?usp=sharing).  You can also run this workbook locally on your device if you already have Python/Jupyter installed, but its not required for today's lecture.

To run a cell, click in the cell and press `Shift` + `Enter`. Try to run the notebook from top to bottom, because later cells often use variables created earlier.

## Big picture

A computational research workflow is more than a calculation. A good workflow keeps track of:

- the scientific question;
- the assumptions or model;
- the input data or simulation settings;
- the code that transforms inputs into outputs;
- the figures, tables, and diagnostics used to interpret the result;
- enough documentation that the work can be rerun later.

This course is about building those habits while also learning practical numerical tools.

## Today's goals

By the end of today's lecture, you should have seen:

- Python as a tool for reproducible calculation;
- a small synthetic data set stored in a table;
- a publication-style plot with uncertainty bars;
- a simple model fit and residual check;
- a preview of Monte Carlo methods;
- why the command line, Git, and project organization matter for research.

## Does the scientific Python stack work?

Run the next cell. The first line tells the notebook to show plots inline. The imports bring in the libraries we will use constantly: `numpy`, `pandas`, `matplotlib`, and pieces of `scipy`.

In [ ]:
%matplotlib inline

import platform
import sys

# Import the standard scientific Python tools for this notebook.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

# Set a plotting style and a fixed random seed for reproducibility.
plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(20260827)

# Print a short environment check.
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print("Scientific Python stack ready.")

## A tiny command-line preview

In Colab and Jupyter notebooks, a line beginning with `!` is sent to the system shell. Next lecture we will use a real terminal, but this gives us a small preview.

In [ ]:
# Ask the shell where the notebook is running.
!pwd
# Check which Python version the notebook is using.
!python --version

# Example 1: A small measurement data set

Suppose we measure a damped oscillation. This could stand in for many real measurements: a mechanical oscillator, a detector response, a signal with decay, or any time-dependent quantity with noise.

We will model the measured position with

$$x(t) = A e^{-\gamma t} \cos(\omega t + \phi) + c,$$

where $A$ is the amplitude, $\gamma$ is the damping constant, $\omega$ is the angular frequency, $\phi$ is the phase, and $c$ is a constant offset.

The data below are synthetic, but the workflow is realistic: define a model, generate or load data, put the data in a table, and inspect the first few rows.

In [ ]:
# Define the physical model we will use to generate and fit data.
def damped_cosine(t, amplitude, damping, angular_frequency, phase, offset):
    return amplitude * np.exp(-damping * t) * np.cos(angular_frequency * t + phase) + offset


# Choose the true parameter values for our synthetic experiment.
true_parameters = {
    "amplitude": 1.25,
    "damping": 0.18,
    "angular_frequency": 2.70,
    "phase": 0.35,
    "offset": 0.05,
}

# Create measurement times, uncertainties, and noisy observations.
time_s = np.linspace(0.0, 8.0, 65)
sigma_m = 0.035 + 0.015 * time_s / time_s.max()
position_m = damped_cosine(time_s, **true_parameters) + rng.normal(0.0, sigma_m)

# Store the generated measurements in a labeled table.
data = pd.DataFrame(
    {
        "time_s": time_s,
        "position_m": position_m,
        "sigma_m": sigma_m,
    }
)

# Preview the first few rows.
data.head()

A table is not just a convenience. Naming columns clearly is part of documenting the calculation. A future reader should not have to guess whether a column is time in seconds, time in nanoseconds, or a sample index.

In [ ]:
# Summarize the numerical columns in the table.
data.describe()

## Plot the data

A first plot is often the fastest way to find mistakes. Plot before you fit. Plot before you believe a number.

In [ ]:
# Create a figure and axes object for the measurement plot.
fig, ax = plt.subplots(figsize=(8, 4.5))

# Plot each measurement with its vertical uncertainty bar.
ax.errorbar(
    data["time_s"],
    data["position_m"],
    yerr=data["sigma_m"],
    fmt="o",
    ms=4,
    capsize=2,
    label="synthetic measurements",
)

# Add labels so the plot can be interpreted on its own.
ax.set_title("Damped oscillation data")
ax.set_xlabel("time (s)")
ax.set_ylabel("position: x(t)")
ax.legend()
plt.show()

### Try this

- Change the random seed in the setup cell and rerun the notebook.
- Increase the measurement uncertainties by changing `sigma_m`.
- Add or remove points by changing the number passed to `np.linspace`.

Each change should make you ask: did the result change for a physical reason, a numerical reason, or just because I changed the simulation?

# Example 2: Fit a model and inspect residuals

In Unit 2 we will spend much more time fitting models to data. Today we only want the outline: choose a model, estimate parameters, and check what the model missed.

In [ ]:
# Give the fitting routine a starting point and allowed parameter range.
initial_guess = [1.0, 0.10, 2.4, 0.0, 0.0]
bounds = ([0.0, 0.0, 0.5, -np.pi, -1.0], [3.0, 1.0, 6.0, np.pi, 1.0])

# Fit the model, using sigma_m as the measurement uncertainty.
best_fit, covariance = curve_fit(
    damped_cosine,
    data["time_s"],
    data["position_m"],
    p0=initial_guess,
    sigma=data["sigma_m"],
    absolute_sigma=True,
    bounds=bounds,
    maxfev=10000,
)

# Convert the covariance matrix into one-sigma parameter uncertainties.
parameter_names = ["amplitude A", "damping gamma", "angular frequency omega", "phase phi", "offset c"]
uncertainties = np.sqrt(np.diag(covariance))

# Put the estimates in a readable table.
fit_table = pd.DataFrame(
    {
        "parameter": parameter_names,
        "estimate": best_fit,
        "standard_uncertainty": uncertainties,
    }
)

fit_table

A fitted parameter without an uncertainty and a diagnostic plot is usually not enough. The next cell compares the fitted curve to the data and plots normalized residuals.

In [ ]:
# Evaluate the fitted model on a smooth grid and at the measured points.
fit_time = np.linspace(data["time_s"].min(), data["time_s"].max(), 400)
fit_curve = damped_cosine(fit_time, *best_fit)
model_at_data = damped_cosine(data["time_s"], *best_fit)
normalized_residuals = (data["position_m"] - model_at_data) / data["sigma_m"]

# Compute a simple goodness-of-fit summary.
degrees_of_freedom = len(data["time_s"]) - len(best_fit)
reduced_chi2 = np.sum(normalized_residuals**2) / degrees_of_freedom

# Make a two-panel diagnostic figure: data above, residuals below.
fig, (ax_data, ax_resid) = plt.subplots(
    2,
    1,
    figsize=(8, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

# Plot the data and best-fit curve.
ax_data.errorbar(
    data["time_s"],
    data["position_m"],
    yerr=data["sigma_m"],
    fmt="o",
    ms=4,
    capsize=2,
    label="data",
)
ax_data.plot(fit_time, fit_curve, lw=2, label="best fit")
ax_data.set_ylabel("position: x(t)")
ax_data.legend()

# Plot residuals in units of each point's uncertainty.
ax_resid.axhline(0.0, color="black", lw=1)
ax_resid.errorbar(data["time_s"], normalized_residuals, yerr=1.0, fmt="o", ms=4, capsize=2)
ax_resid.set_xlabel("time (s)")
ax_resid.set_ylabel("resid. / sigma")

# Finish the figure layout.
fig.suptitle(f"Model fit with reduced chi^2 = {reduced_chi2:.2f}")
fig.tight_layout()
plt.show()

### What to notice

- The fit returns numbers, but the plot tells us whether those numbers describe the data.
- Residuals should not show an obvious pattern if the model and uncertainty estimates are reasonable.
- A complete analysis should say what was fit, what assumptions were made, and what checks were performed.

# Example 3: A numerical simulation

In Unit 3 we will solve differential equations and study numerical error. Here is a first look at using `scipy` to integrate a damped oscillator equation:

$$\frac{d^2x}{dt^2} + 2\gamma \frac{dx}{dt} + \omega_0^2 x = 0.$$

For the underdamped case, $\gamma < \omega_0$, the analytical solution is

$$x(t) = e^{-\gamma t}\left[x_0 \cos(\omega_d t) + \frac{v_0 + \gamma x_0}{\omega_d}\sin(\omega_d t)\right],$$

where $\omega_d = \sqrt{\omega_0^2 - \gamma^2}$. We will compare this known solution with the numerical result.

In [ ]:
# Write the second-order oscillator equation as two first-order equations.
def oscillator_rhs(t, state, damping, natural_frequency):
    x, v = state
    return [v, -2.0 * damping * v - natural_frequency**2 * x]


# Analytical solution for the underdamped case with initial position x0 and velocity v0.
def oscillator_analytic(t, damping, natural_frequency, x0, v0):
    damped_frequency = np.sqrt(natural_frequency**2 - damping**2)
    cosine_part = x0 * np.cos(damped_frequency * t)
    sine_part = (v0 + damping * x0) / damped_frequency * np.sin(damped_frequency * t)
    return np.exp(-damping * t) * (cosine_part + sine_part)


# Choose model parameters, initial conditions, and output times.
damping_demo = 0.12
natural_frequency_demo = 2.4
initial_state = [1.0, 0.0]
t_eval = np.linspace(0.0, 12.0, 600)

# Numerically integrate the differential equation.
solution = solve_ivp(
    oscillator_rhs,
    t_span=(t_eval.min(), t_eval.max()),
    y0=initial_state,
    t_eval=t_eval,
    args=(damping_demo, natural_frequency_demo),
    rtol=1e-8,
    atol=1e-10,
)

# Extract the numerical position and compute the analytical benchmark.
x_num, v_num = solution.y
x_exact = oscillator_analytic(
    solution.t,
    damping_demo,
    natural_frequency_demo,
    initial_state[0],
    initial_state[1],
)
max_abs_difference = np.max(np.abs(x_num - x_exact))

# Plot the numerical and analytical solutions on the same axes.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(solution.t, x_num, lw=2, label="numerical solve_ivp")
ax.plot(solution.t, x_exact, "--", lw=2, label="analytical solution")
ax.set_title("Damped oscillator: numerical vs. analytical")
ax.set_xlabel("time (s)")
ax.set_ylabel("position: x(t)")
ax.legend()

fig.tight_layout()
plt.show()

print(f"Maximum absolute difference: {max_abs_difference:.2e}")

This is a small example, but it already raises research questions: How accurate is the integrator? How should we choose tolerances? When an analytical solution is not available, what benchmark or convergence test should we use instead?

# Example 4: Monte Carlo estimation

In Unit 4 we will use random sampling for simulation, uncertainty propagation, and inference. The classic first example is estimating $\pi$ by throwing random points into a square and counting how many land inside a circle.

In [ ]:
# Use a separate random generator for the Monte Carlo example.
mc_rng = np.random.default_rng(427)

# Throw random points into the square [-1, 1] x [-1, 1].
n_points = 4000
points = mc_rng.uniform(-1.0, 1.0, size=(n_points, 2))
x = points[:, 0]
y = points[:, 1]

# Count how many points land inside the unit circle.
radius_squared = x**2 + y**2
inside = radius_squared <= 1.0
pi_estimate = 4.0 * inside.mean()

# Plot y vs. x for the random points.
fig, ax = plt.subplots(figsize=(5.5, 5.5))

# Separate points inside and outside the unit circle.
ax.scatter(x[inside], y[inside], s=4, alpha=0.45, label="inside circle")
ax.scatter(x[~inside], y[~inside], s=4, alpha=0.45, label="outside circle")
circle = plt.Circle((0.0, 0.0), 1.0, fill=False, color="black", lw=1.5)
ax.add_patch(circle)
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"Monte Carlo estimate using {n_points} random points")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(markerscale=3)

fig.tight_layout()
plt.show()

# Print the headline numerical result.
print(f"Monte Carlo estimate with {n_points} points: pi = {pi_estimate:.5f}")
print(f"Reference value from NumPy:              pi = {np.pi:.5f}")

Now repeat the same estimate for increasing sample sizes. The method is the same, but we track how the answer changes as we throw more random points.

In [ ]:
# Use a fresh random generator for the convergence study.
convergence_rng = np.random.default_rng(428)

# Choose sample sizes spread across several powers of ten.
n_values = np.unique(np.logspace(2, 5, 25, dtype=int))
pi_estimates = []

# Store one pi estimate for each sample size.
for n in n_values:
    trial_points = convergence_rng.uniform(-1.0, 1.0, size=(n, 2))
    trial_radius_squared = trial_points[:, 0] ** 2 + trial_points[:, 1] ** 2
    trial_inside = trial_radius_squared <= 1.0
    pi_estimates.append(4.0 * trial_inside.mean())

# Compare Monte Carlo estimates to the reference value of pi.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.semilogx(n_values, pi_estimates, "o-", label="Monte Carlo estimate")
ax.axhline(np.pi, color="black", lw=1, label="np.pi")
ax.set_xlabel("number of samples")
ax.set_ylabel("estimate of pi")
ax.set_title("Monte Carlo convergence is noisy")
ax.legend()

fig.tight_layout()
plt.show()

### Try this

- In the first Monte Carlo cell, increase `4000` to `40000` and rerun.
- Change one of the random seeds and rerun.
- In the second Monte Carlo cell, look at how the estimate changes with sample size.

Randomness can be useful, but only if we measure and communicate the uncertainty that comes with it.

# Why workflow matters

The code above was intentionally small, but a research version of this work quickly grows into multiple files and decisions:

- Which data are raw, and which are processed?
- Which script created each figure?
- Which parameter values were used?
- Which environment and package versions were used?
- Which results are final, and which were exploratory?
- How can a collaborator rerun the calculation?

This is why Unit 1 starts with command-line work, Git, Python environments, project organization, and documentation.

# Preparation for command-line, etc. on your laptop and beyond

Starting next week, we will be using command-line (shell) tools to begin writing and executing your own code.  If you already have your preferred tools for navigating the command line, editing and compiling code, etc. you're welcome to use those.  I will be using a popular tool called Visual Studio Code (VS Code) for in-class demonstrations, which you're welcome to utilize as well.  If you'd like to follow along in VS Code, install it before the next lecture so we can use a native terminal, command-line tools, Git, and project folders on your own machine.

- Course resource: [Visual Studio Code (VS Code)](../resources/vscode.md)
- GitHub version: [Visual Studio Code (VS Code)](https://github.com/jrstevenjlab/gradcompphys/blob/main/resources/vscode.md)

Next time we will move from notebook cells toward a more complete research project structure.